# Introduction to Sliced Inference with SAHI

[SAHI](https://github.com/obss/sahi) (Slicing Aided Hyper Inference) solves a fundamental problem in aerial and satellite object detection: standard detectors are trained on natural images at 640×640 px and struggle to find small objects (vehicles, pedestrians, utility equipment) that occupy only a handful of pixels in a large scene.

**How SAHI works:**
1. Tile the image into overlapping patches (e.g., 640×640 px with 20% overlap)
2. Run any YOLO / DETR detector on each patch independently
3. Map detections back to the original image coordinates
4. Merge overlapping boxes with NMS

This typically improves small-object AP by **5–7%** with no model retraining.

## References
- GitHub: https://github.com/obss/sahi
- Paper: https://ieeexplore.ieee.org/document/9897990
- Docs: https://docs.obss.ai/sahi/

## 1. Verify Installation

In [ ]:
import sahi
print(f"SAHI version: {sahi.__version__}")

## 2. Load a Detection Model

`AutoDetectionModel` wraps any supported backend (Ultralytics YOLO, MMDetection, Detectron2, HuggingFace, etc.) behind a unified API.

In [ ]:
from sahi import AutoDetectionModel

# YOLOv8n weights are downloaded automatically on first use (~6 MB)
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="yolov8n.pt",
    confidence_threshold=0.3,
    device="cpu",  # change to "cuda:0" if GPU is available
)
print("Model loaded.")

## 3. Download a Sample Large Image

We use a sample aerial image. In practice, this could be an orthophoto or a satellite scene.

In [ ]:
import os
import urllib.request

# DOTA-style aerial image with small vehicles
url = "https://raw.githubusercontent.com/obss/sahi/main/demo/demo_data/coco_utils/terrain2_coco.jpg"
image_path = "sample_aerial.jpg"
if not os.path.exists(image_path):
    urllib.request.urlretrieve(url, image_path)

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread(image_path)
print(f"Image shape: {img.shape}")
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.title("Sample aerial image")
plt.tight_layout()
plt.show()

## 4. Standard Inference (no slicing)

In [ ]:
from sahi.predict import get_prediction

result_standard = get_prediction(
    image=image_path,
    detection_model=detection_model,
)

print(f"Standard inference detections: {len(result_standard.object_prediction_list)}")

## 5. Sliced Inference

In [ ]:
from sahi.predict import get_sliced_prediction

result_sliced = get_sliced_prediction(
    image=image_path,
    detection_model=detection_model,
    slice_height=320,
    slice_width=320,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
)

print(f"Sliced inference detections: {len(result_sliced.object_prediction_list)}")

## 6. Visualize and Compare

In [ ]:
result_standard.export_visuals(export_dir=".", file_name="standard_result")
result_sliced.export_visuals(export_dir=".", file_name="sliced_result")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(mpimg.imread("standard_result.png"))
axes[0].set_title(f"Standard  ({len(result_standard.object_prediction_list)} detections)")
axes[0].axis("off")
axes[1].imshow(mpimg.imread("sliced_result.png"))
axes[1].set_title(f"Sliced  ({len(result_sliced.object_prediction_list)} detections)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 7. Supported Backends

SAHI works with many detectors out of the box:

| `model_type` | Backend | Install |
|---|---|---|
| `ultralytics` | YOLOv8/v9/v10/v11 | `pip install ultralytics` |
| `mmdet` | MMDetection | `pip install mmdet` |
| `detectron2` | Detectron2 | `pip install detectron2` |
| `huggingface` | Transformers DETR/DETA | `pip install transformers` |
| `torchvision` | Faster R-CNN, RetinaNet | `pip install torchvision` |

Switch backends by changing `model_type` in `AutoDetectionModel.from_pretrained()` — the slicing API stays identical.